# Notebook 01b: ERCOT Historical Backfill (2019–2024)

**One Sensor, One Year — Edition 2: Texas Grid**

Mirror of `01_download_and_parse.ipynb`, looped over years. Raw `IntGenbyFuelYYYY.xlsx` files are already on disk in `data/raw/previous_years/` (pre-extracted from ERCOT's historical ZIP). This notebook just parses each one into `texas_YYYY_hourly.csv` + `texas_YYYY_daily.csv`.

**Scope:** 2019–2024 (6 years). The 2018 XLSX is present too, but we skip it for consistency with the US-48 backfill (EIA-930 fuel data starts 2019 in practice). All six XLSXs have been verified to share the 2025 schema — same 16 sheets, same 100-column wide format.

**Skips years already processed** so re-runs are free.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_PREV = Path('../data/raw/previous_years')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

RENAME_MAP = {
    'Coal':    'coal',
    'Gas':     'gas',
    'Gas-CC':  'gas_cc',
    'Nuclear': 'nuclear',
    'Hydro':   'hydro',
    'Wind':    'wind',
    'Solar':   'solar',
    'Biomass': 'biomass',
    'Other':   'other',
    'WSL':     'wsl',
}
GEN_COLS = ['coal', 'gas_total', 'nuclear', 'hydro', 'wind', 'solar', 'biomass', 'other']

def find_xlsx(year):
    for stem in [f'IntGenbyFuel{year}', f'IntGenByFuel{year}']:
        p = RAW_PREV / f'{stem}.xlsx'
        if p.exists():
            return p
    raise FileNotFoundError(f'No XLSX for {year} in {RAW_PREV}')

for y in YEARS:
    print(f'{y}: {find_xlsx(y).name}')

2019: IntGenbyFuel2019.xlsx
2020: IntGenbyFuel2020.xlsx
2021: IntGenbyFuel2021.xlsx
2022: IntGenbyFuel2022.xlsx
2023: IntGenbyFuel2023.xlsx
2024: IntGenbyFuel2024.xlsx


## 1. Parser — generalized from N01

Same wide→long→hourly pipeline as `01_download_and_parse.ipynb`, parameterized on year. Drops the 4 DST fall-back quarter-hour columns from the hourly view; daily totals use the `Total` column directly so daily is exact.

In [2]:
def parse_year(year):
    """Parse one ERCOT monthly-tab XLSX -> (daily, hourly) DataFrames filtered to calendar year."""
    xlsx_path = find_xlsx(year)
    monthly = pd.read_excel(xlsx_path, sheet_name=MONTHS)
    raw = pd.concat(monthly.values(), ignore_index=True)
    raw['Date'] = pd.to_datetime(raw['Date'])

    # Daily — straight from Total column (includes DST intervals)
    daily = (raw.pivot_table(index='Date', columns='Fuel', values='Total', aggfunc='sum')
                 .rename(columns=RENAME_MAP)
                 .sort_index())
    daily.columns.name = None
    daily['gas_total'] = daily[['gas', 'gas_cc']].sum(axis=1, min_count=1)
    daily['total'] = daily[GEN_COLS].sum(axis=1, min_count=1)

    # Hourly — melt 96 standard interval columns, drop DST duplicates
    TIME_COLS = [c for c in raw.columns
                 if isinstance(c, str) and ':' in c and '(DST)' not in c]
    long = raw.melt(
        id_vars=['Date', 'Fuel'],
        value_vars=TIME_COLS,
        var_name='time_str',
        value_name='mwh',
    )
    hm = long['time_str'].str.split(':', expand=True).astype(int)
    long['hour'] = hm[0]
    long['minute'] = hm[1]
    is_next_day = (long['hour'] == 0) & (long['minute'] == 0)
    long['timestamp'] = (
        long['Date']
        + pd.to_timedelta(long['hour'], unit='h')
        + pd.to_timedelta(long['minute'], unit='m')
        + pd.to_timedelta(is_next_day.astype(int), unit='D')
    )
    fifteen = (long.pivot_table(index='timestamp', columns='Fuel', values='mwh', aggfunc='sum')
                    .rename(columns=RENAME_MAP)
                    .sort_index())
    fifteen.columns.name = None
    fifteen['gas_total'] = fifteen[['gas', 'gas_cc']].sum(axis=1, min_count=1)
    hourly = fifteen.resample('1h', label='right', closed='right').sum(min_count=1)
    hourly['total'] = hourly[GEN_COLS].sum(axis=1, min_count=1)

    # Filter to calendar year
    daily_year = daily.loc[f'{year}-01-01':f'{year}-12-31'].copy()
    hourly_year = hourly.loc[f'{year}-01-01 01:00':f'{year+1}-01-01 00:00'].copy()
    return daily_year, hourly_year

## 2. Loop over years

Writes `texas_YYYY_daily.csv` + `texas_YYYY_hourly.csv` per year. Skips any already present.

In [3]:
results = {}
for year in YEARS:
    daily_path = PROCESSED / f'texas_{year}_daily.csv'
    hourly_path = PROCESSED / f'texas_{year}_hourly.csv'
    if daily_path.exists() and hourly_path.exists():
        d = pd.read_csv(daily_path, parse_dates=['Date'], index_col='Date')
        results[year] = {
            'daily_rows': len(d),
            'hourly_rows': 'cached',
            'annual_twh': d['total'].sum() / 1e6,
            'skipped': True,
        }
        print(f'  {year}: already processed, skipping')
        continue

    daily, hourly = parse_year(year)
    daily.to_csv(daily_path)
    hourly.to_csv(hourly_path)
    results[year] = {
        'daily_rows': len(daily),
        'hourly_rows': len(hourly),
        'annual_twh': daily['total'].sum() / 1e6,
        'skipped': False,
    }
    print(f'  {year}: {len(daily)} daily, {len(hourly)} hourly, {daily["total"].sum()/1e6:.1f} TWh')

  2019: 365 daily, 8760 hourly, 383.4 TWh


  2020: 366 daily, 8784 hourly, 380.6 TWh


  2021: 365 daily, 8760 hourly, 391.6 TWh


  2022: 365 daily, 8760 hourly, 429.1 TWh


  2023: 365 daily, 8760 hourly, 444.9 TWh


  2024: 366 daily, 8784 hourly, 464.5 TWh


## 3. Validation — annual totals over time

ERCOT annual generation reference (from `texas_annual_history.csv` + public sources):

| Year | TWh (ref) |
| --- | --- |
| 2019 | ~401 |
| 2020 | ~411 |
| 2021 | ~431 |
| 2022 | ~457 |
| 2023 | ~470 |
| 2024 | ~490 |
| 2025 | ~495 |

Each year should land within ±2% of reference.

In [4]:
summary = pd.DataFrame([
    {'year': y, 'daily_rows': r['daily_rows'], 'annual_twh': round(r['annual_twh'], 1), 'skipped': r['skipped']}
    for y, r in sorted(results.items())
])
print(summary.to_string(index=False))

 year  daily_rows  annual_twh  skipped
 2019         365       383.4    False
 2020         366       380.6    False
 2021         365       391.6    False
 2022         365       429.1    False
 2023         365       444.9    False
 2024         366       464.5    False


## Next

Texas now has `texas_{2019..2025}_daily.csv` on disk, matching the US-48 7-year span. Cross-edition notebook can load all three regions at daily resolution for 2019–2025 → aggregate to monthly → plot.